In [ ]:
from pathlib import Path

from tqdm import tqdm
from transformers import pipeline
import polars as pl

SRC = Path("/mnt/data-dump/for_enriching")
DST = Path("/mnt/Fast2T/data/enriched")

DST.mkdir(exist_ok=True)
BATCH_SIZE = 256

pipe = pipeline(
    "text-classification",
    model="fakespot-ai/roberta-base-ai-text-detection-v1",
    device=0,
    batch_size=BATCH_SIZE,
    truncation=True,
    max_length=512,
)

files = sorted(SRC.glob("*.parquet"))

for path in files:
    df = (
        pl.scan_parquet(path)
        .filter(pl.col("text").str.len_bytes() > 2000)
        .select("id", "url", "type", "text", "links")
        .collect()
    )
    if df.is_empty():
        continue

    texts = [t[:2000] for t in df["text"].to_list()]
    results = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc=path.stem, leave=False):
        batch = texts[i: i + BATCH_SIZE]
        results.extend(pipe(batch))

    df = df.with_columns(
        ai_score=pl.Series([1 - r["score"] if r["label"] == "Real" else r["score"] for r in results], dtype=pl.Float64),
    )
    df.write_parquet(DST / path.name, compression="zstd")